# Fine-tuning QLoRA — PubMedQA (Llama-3-8B / Mistral-7B)


Este notebook treina um modelo biomédico usando **Unsloth (QLoRA 4-bit)** no dataset **PubMedQA**.

**Antes de rodar:**
1. Menu `Ambiente de execução` → `Alterar tipo de ambiente de execução` → GPU → **T4**
2. Rode as células em ordem, de cima para baixo
3. O treino é controlado por `MAX_STEPS` (não por épocas), então o tempo é previsível (~15-25 min para 60 steps)


## 1. Instalação das dependências

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes datasets


## 2. Montar o Google Drive (para salvar checkpoints)

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Configurações gerais
Ajuste os parâmetros abaixo conforme necessidade.

In [3]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Modelo base
MODEL_NAME = "unsloth/llama-3-8b-bnb-4bit"   # alternativa: "unsloth/mistral-7b-bnb-4bit"

# Dataset: "pqa_labeled" (1k exemplos, recomendado para começar rápido)
#          "pqa_artificial" (211k exemplos — use N_SAMPLES para limitar)
DATASET_SPLIT = "pqa_labeled"
N_SAMPLES = None  # ex: 3000, se usar "pqa_artificial". None = usa todos do split.

# Comprimento máximo de sequência
MAX_SEQ_LENGTH = 1024

# Quantização 4-bit (obrigatório para caber no T4 com modelo 7B/8B)
LOAD_IN_4BIT = True

# LoRA
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0

# Treino — controle direto por passos (não por épocas)
MAX_STEPS = 60          # ajuste conforme tempo disponível
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4    # batch efetivo = BATCH_SIZE * GRAD_ACCUM_STEPS = 8
LEARNING_RATE = 2e-4
WARMUP_STEPS = 5
LOGGING_STEPS = 5
SAVE_STEPS = 20

# Saída
OUTPUT_DIR = "/content/drive/MyDrive/medllm_checkpoints"
FINAL_MODEL_DIR = "/content/drive/MyDrive/medllm_final"

SEED = 3407


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 4. Carregar modelo base em 4-bit + aplicar LoRA

In [4]:
print(f"Carregando modelo base: {MODEL_NAME}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,  # detecta automaticamente (fp16 no T4)
    load_in_4bit=LOAD_IN_4BIT,
)

print("Aplicando adaptadores LoRA")
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)


Carregando modelo base: unsloth/llama-3-8b-bnb-4bit
==((====))==  Unsloth 2026.8.22: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


Aplicando adaptadores LoRA


Unsloth 2026.8.22 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 5. Carregar e formatar o dataset PubMedQA (formato Alpaca)

In [5]:
ALPACA_PROMPT = """Below is a medical question with context. Provide a well-reasoned answer.

### Context:
{}

### Question:
{}

### Answer:
{}"""

def format_example(example, eos_token):
    contexts = example.get("context", {}).get("contexts", [])
    context_text = " ".join(contexts) if contexts else "No context provided."    question = example["question"]
    long_answer = example.get("long_answer", "")
    final_decision = example.get("final_decision", "")

    answer = f"{long_answer}\n\nConclusion: {final_decision}".strip()
    text = ALPACA_PROMPT.format(context_text, question, answer) + eos_token
    return {"text": text}


print(f"Carregando dataset PubMedQA (split: {DATASET_SPLIT})")
raw_dataset = load_dataset("qiaojin/PubMedQA", DATASET_SPLIT, split="train")

if N_SAMPLES is not None:
    raw_dataset = raw_dataset.select(range(min(N_SAMPLES, len(raw_dataset))))

print(f"Total de exemplos usados: {len(raw_dataset)}")

eos_token = tokenizer.eos_token
dataset = raw_dataset.map(
    lambda ex: format_example(ex, eos_token),
    remove_columns=raw_dataset.column_names,
)

# Conferir um exemplo formatado
print(dataset[0]["text"])


Carregando dataset PubMedQA (split: pqa_labeled)
Total de exemplos usados: 1000
Abaixo está uma pergunta médica com contexto. Responda de forma fundamentada.

### Contexto:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants. The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the progression of PCD; cells that will not undergo P

## 6. Treino
Controlado por `MAX_STEPS` — para exatamente quando atingir o número de passos definido acima.

In [6]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    warmup_steps=WARMUP_STEPS,
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not torch.cuda.is_bf16_supported(),  # T4 não suporta bf16
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=LOGGING_STEPS,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    output_dir=OUTPUT_DIR,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    report_to="none",  # evita travar esperando login do W&B
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    args=training_args,
)

trainer_stats = trainer.train()
print(f"Treino concluído em {trainer_stats.metrics['train_runtime']:.1f}s")


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
5,1.879740
10,1.659211
15,1.561341
20,1.506995
25,1.494294
30,1.508819
35,1.497482
40,1.507683
45,1.520017
50,1.516191


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/medllm_checkpoints/checkpoint-20/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/medllm_checkpoints/checkpoint-40/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/medllm_checkpoints/checkpoint-60/tokenizer_config.json.


Treino concluído em 989.4s


## 7. Salvar modelo final (adaptadores LoRA)

In [7]:
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)
model.save_pretrained(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print(f"Modelo salvo em: {FINAL_MODEL_DIR}")


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/medllm_final/tokenizer_config.json.


Modelo salvo em: /content/drive/MyDrive/medllm_final


## 8. Exportar para GGUF e publicar no Hugging Face Hub

Necessário para rodar depois localmente via **Ollama**. Substitua `seu-usuario/nome-do-modelo` e informe seu token do HF.

In [12]:
from huggingface_hub import login

from google.colab import userdata
login(token=userdata.get("HF_TOKEN"))

model.push_to_hub_gguf(
    "jeferson2106/medico_ia_jeferson",
    tokenizer,
    quantization_method="q4_k_m",
)


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_pi19_0ak/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uf/llama-3-8b.Q4_K_M.gguf:   0%|          | 14.0MB / 4.92GB            

Uploading config.json...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/jeferson2106/medico_ia_jeferson
Unsloth: Cleaning up temporary files...


'jeferson2106/medico_ia_jeferson'

## 9. Teste rápido de inferência no próprio Colab

Útil para validar a qualidade antes de exportar.

In [8]:
FastLanguageModel.for_inference(model)

pergunta_teste = "Does metformin reduce cardiovascular risk in diabetic patients?"
contexto_teste = "Several observational studies and clinical trials have evaluated the effect of metformin on cardiovascular outcomes in patients with type 2 diabetes."

prompt = ALPACA_PROMPT.format(contexto_teste, pergunta_teste, "")
inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=200, use_cache=True)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])


Both `max_new_tokens` (=200) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Abaixo está uma pergunta médica com contexto. Responda de forma fundamentada.

### Contexto:
Several observational studies and clinical trials have evaluated the effect of metformin on cardiovascular outcomes in patients with type 2 diabetes.

### Pergunta:
Does metformin reduce cardiovascular risk in diabetic patients?

### Resposta:
Metformin appears to be associated with a reduction in cardiovascular risk in patients with type 2 diabetes. However, the evidence is based on observational studies and clinical trials with methodological limitations. Therefore, the cardiovascular benefits of metformin cannot be definitively established. Further studies are required to confirm the cardiovascular benefits of metformin in patients with type 2 diabetes.

Conclusão: yes


## 10. Integração com LangChain e fluxo de decisão com LangGraph (segurança e validação)

Conecta o modelo fine-tuned a um pipeline LangChain que consulta uma base estruturada simulada de prontuários, contextualiza a resposta com dados do paciente e organiza o atendimento em um fluxo automatizado com LangGraph (verificar exames pendentes, emitir alertas, sugerir conduta). Inclui guardrails contra prescrição direta, logging de auditoria e explainability (fonte da resposta).

In [9]:
!pip install -q langchain langchain-community langchain-huggingface langgraph
# Reinstala a versão de 'requests' exigida pelo google-colab, sobrescrita pelo langchain acima
!pip install -q "requests==2.32.4"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [10]:
import json
import logging
from datetime import datetime

# Log de auditoria: cada evento do fluxo é registrado com timestamp para rastreamento
LOG_PATH = "/content/drive/MyDrive/medllm_checkpoints/assistente_auditoria.log"

logging.basicConfig(
    filename=LOG_PATH,
    level=logging.INFO,
    format="%(asctime)s | %(message)s",
)
logger_auditoria = logging.getLogger("assistente_medico")

def registrar_log(etapa: str, dados: dict) -> dict:
    """Registra um evento do fluxo para auditoria e rastreamento (requisito de segurança)."""
    registro = {"timestamp": datetime.utcnow().isoformat(), "etapa": etapa, **dados}
    logger_auditoria.info(json.dumps(registro, ensure_ascii=False, default=str))
    return registro

# Termos que indicam prescrição/conduta direta -- exigem validação humana explícita
TERMOS_PRESCRICAO = ["take ", "administer ", "i prescribe ", "prescription for ", "dose of ", "apply ", "tome ", "administre ", "prescrevo ", "receita de ", "dose de ", "aplique "]
def aplicar_guardrails(resposta: str) -> str:
    """
    Limite de atuação do assistente: nunca prescrever diretamente sem validação humana.
    Filtro simples baseado em palavras-chave; em produção, substituir por um classificador dedicado.
    """
    resposta_lower = resposta.lower()
    contem_prescricao = any(termo in resposta_lower for termo in TERMOS_PRESCRICAO)
    if contem_prescricao:
        aviso = (
            "\n\nAVISO DE SEGURANÇA: esta resposta sugere uma conduta ou medicação. "
            "O assistente NÃO substitui o julgamento clínico -- toda prescrição exige "
            "validação e assinatura de um médico responsável antes de ser aplicada ao paciente."
        )
        resposta = resposta + aviso
    registrar_log("guardrail_aplicado", {"prescricao_detectada": contem_prescricao})
    return resposta

In [15]:
import pandas as pd
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from transformers import pipeline as hf_pipeline

# --- Wrapper do modelo fine-tuned como LLM do LangChain ---
FastLanguageModel.for_inference(model)

gerador = hf_pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=300,
    do_sample=False,
    temperature=0.1,
    return_full_text=False,
)
llm = HuggingFacePipeline(pipeline=gerador)

# --- Base de dados estruturada simulada (prontuários eletrônicos) ---
# Em produção, isso viria do sistema hospitalar (HL7/FHIR, banco relacional, etc.).
# Aqui usamos dados sintéticos para demonstrar o fluxo, sem expor dados reais de pacientes.
prontuarios_df = pd.DataFrame([
    {
        "paciente_id": "P001",
        "idade": 62,
        "condicoes": "Diabetes tipo 2, Hipertensão",
        "exames_pendentes": ["Hemoglobina glicada (HbA1c)", "Perfil lipídico"],
        "exames_realizados": {"Glicemia de jejum": "182 mg/dL", "Creatinina": "1.1 mg/dL"},
        "medicacoes_atuais": ["Metformina 850mg 2x/dia"],
    },
    {
        "paciente_id": "P002",
        "idade": 45,
        "condicoes": "Asma",
        "exames_pendentes": [],
        "exames_realizados": {"Espirometria": "Normal"},
        "medicacoes_atuais": ["Salbutamol conforme necessidade"],
    },
])

def consultar_prontuario(paciente_id: str) -> dict:
    """Tool de consulta ao prontuário estruturado do paciente (requisito: consultas em base estruturada)."""
    linha = prontuarios_df[prontuarios_df["paciente_id"] == paciente_id]
    return linha.iloc[0].to_dict() if not linha.empty else {}

# --- Prompt contextualizado com dados do paciente + exigência de explainability ---
ASSISTENTE_PROMPT = PromptTemplate.from_template(
    "You are a virtual medical assistant for the hospital. Answer based only on the "
    "internal protocol and the patient data below. At the end, state the source of "
    "the information used (internal protocol, patient record, or general clinical knowledge).\n\n"
    "### Protocol / clinical context:\n{contexto}\n\n"
    "### Current patient data:\n{dados_paciente}\n\n"
    "### Doctor's question:\n{pergunta}\n\n"
    "### Answer (end with 'Source:'):\n"
)
def responder_com_contexto(pergunta: str, paciente_id: str, contexto_clinico: str = "No additional protocol provided.") -> dict:
    """Pipeline LangChain: consulta o prontuário, contextualiza o prompt e aplica os guardrails de segurança."""
    dados_paciente = consultar_prontuario(paciente_id)
    registrar_log("consulta_prontuario", {"paciente_id": paciente_id, "encontrado": bool(dados_paciente)})

    prompt = ASSISTENTE_PROMPT.format(
        contexto=contexto_clinico,
        dados_paciente=json.dumps(dados_paciente, ensure_ascii=False, default=str),
        pergunta=pergunta,
    )
    resposta_bruta = llm.invoke(prompt)
    resposta_validada = aplicar_guardrails(resposta_bruta)

    registrar_log("resposta_gerada", {"paciente_id": paciente_id, "pergunta": pergunta})
    return {"resposta": resposta_validada, "dados_paciente": dados_paciente}

In [16]:
from typing import TypedDict, List, Optional
from langgraph.graph import StateGraph, START, END

class EstadoAtendimento(TypedDict):
    paciente_id: str
    pergunta: str
    dados_paciente: dict
    exames_pendentes: List[str]
    resposta_llm: Optional[str]
    alertas: List[str]

def no_receber_paciente(estado: EstadoAtendimento) -> dict:
    dados = consultar_prontuario(estado["paciente_id"])
    registrar_log("no_receber_paciente", {"paciente_id": estado["paciente_id"]})
    return {"dados_paciente": dados}

def no_verificar_exames(estado: EstadoAtendimento) -> dict:
    pendentes = estado["dados_paciente"].get("exames_pendentes", [])
    registrar_log("no_verificar_exames", {"pendentes": pendentes})
    return {"exames_pendentes": pendentes}

def no_emitir_alertas(estado: EstadoAtendimento) -> dict:
    alertas = [f"Exame pendente: {exame}" for exame in estado["exames_pendentes"]]
    registrar_log("no_emitir_alertas", {"alertas": alertas})
    return {"alertas": alertas}

def no_sugerir_conduta(estado: EstadoAtendimento) -> dict:
    resultado = responder_com_contexto(estado["pergunta"], estado["paciente_id"])
    registrar_log("no_sugerir_conduta", {"paciente_id": estado["paciente_id"]})
    return {"resposta_llm": resultado["resposta"]}

def rota_apos_exames(estado: EstadoAtendimento) -> str:
    """Fluxo de decisão: paciente com exames pendentes recebe alerta antes da sugestão de conduta."""
    return "com_pendencia" if estado["exames_pendentes"] else "sem_pendencia"

grafo = StateGraph(EstadoAtendimento)
grafo.add_node("receber_paciente", no_receber_paciente)
grafo.add_node("verificar_exames", no_verificar_exames)
grafo.add_node("emitir_alertas", no_emitir_alertas)
grafo.add_node("sugerir_conduta", no_sugerir_conduta)

grafo.add_edge(START, "receber_paciente")
grafo.add_edge("receber_paciente", "verificar_exames")
grafo.add_conditional_edges(
    "verificar_exames",
    rota_apos_exames,
    {"com_pendencia": "emitir_alertas", "sem_pendencia": "sugerir_conduta"},
)
grafo.add_edge("emitir_alertas", "sugerir_conduta")
grafo.add_edge("sugerir_conduta", END)

fluxo_assistente = grafo.compile()
print("Grafo LangGraph compilado com sucesso.")

Grafo LangGraph compilado com sucesso.


In [17]:
estado_inicial = {
    "paciente_id": "P001",
    "pergunta": "What treatment approach is recommended for this patient's glycemic control?",
    "dados_paciente": {},
    "exames_pendentes": [],
    "resposta_llm": None,
    "alertas": [],
}

resultado_final = fluxo_assistente.invoke(estado_inicial)

print("Alertas emitidos:")
for alerta in resultado_final["alertas"]:
    print(f" - {alerta}")

print("\nResposta do assistente médico:\n")
print(resultado_final["resposta_llm"])

print(f"\nLog de auditoria salvo em: {LOG_PATH}")

/tmp/ipykernel_63431/1077327527.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  registro = {"timestamp": datetime.utcnow().isoformat(), "etapa": etapa, **dados}
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Alertas emitidos:
 - Exame pendente: Hemoglobina glicada (HbA1c)
 - Exame pendente: Perfil lipídico

Resposta do assistente médico:

Metformin monotherapy is recommended for this patient. Source: patient record.

Conclusão: yes

Log de auditoria salvo em: /content/drive/MyDrive/medllm_checkpoints/assistente_auditoria.log


/tmp/ipykernel_63431/1077327527.py:17: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  registro = {"timestamp": datetime.utcnow().isoformat(), "etapa": etapa, **dados}


## 11. Diagrama do fluxo (para o relatório técnico)

Gera a representação do grafo LangGraph em Mermaid (diagrama do fluxo LangChain/LangGraph).

In [18]:
# Gera a representação Mermaid do grafo -- útil para colar no relatório técnico da Fase 3
print(fluxo_assistente.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	receber_paciente(receber_paciente)
	verificar_exames(verificar_exames)
	emitir_alertas(emitir_alertas)
	sugerir_conduta(sugerir_conduta)
	__end__([<p>__end__</p>]):::last
	__start__ --> receber_paciente;
	emitir_alertas --> sugerir_conduta;
	receber_paciente --> verificar_exames;
	verificar_exames -. &nbsp;com_pendencia&nbsp; .-> emitir_alertas;
	verificar_exames -. &nbsp;sem_pendencia&nbsp; .-> sugerir_conduta;
	sugerir_conduta --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

